# 03B 模型验证与调参：这个高分可信吗？

> 🟢 **Level A · 必须掌握** | 完成标准：解释 `train → CV/tune → untouched test`。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,cross_validate,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].fillna(df[features].median()); y=df['CO2-1 bar (mol/kg)']
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)
m=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)
s=cross_validate(m,X_dev,y_dev,cv=cv,scoring={'r2':'r2','mae':'neg_mean_absolute_error'})
print('CV R² =',s['test_r2'].mean(),'+/-',s['test_r2'].std())
print('CV MAE =',(-s['test_mae']).mean(),'+/-',(-s['test_mae']).std())


In [ ]:
param={'n_estimators':[150,300,500],'max_depth':[None,6,10,16],'min_samples_leaf':[1,2,4],'max_features':['sqrt',0.7,1.0]}
search=RandomizedSearchCV(RandomForestRegressor(random_state=42,n_jobs=-1),param,n_iter=12,cv=cv,scoring='neg_mean_absolute_error',random_state=42,n_jobs=-1).fit(X_dev,y_dev)
p=search.best_estimator_.predict(X_test)
print(search.best_params_)
print('test MAE =',mean_absolute_error(y_test,p),'test R² =',r2_score(y_test,p))


## 常见 leakage
near-duplicates 跨 split、family/topology 泄漏、split 前在全数据上 preprocessing/selection、target-derived feature、调参时反复看 test。
